In [1]:
import os
from pathlib import Path
import yaml
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models



In [2]:
# ==========================================
# 1. Automatic Path Resolution (The Fix)
# ==========================================
# Move up directories until we find the 'data' folder
while not Path("data").exists() and len(Path.cwd().parts) > 1:
    os.chdir("..")
print(f"✅ Working directory set to: {os.getcwd()}")


✅ Working directory set to: E:\CS\SEM 6\DL\Mini Project\Trash-Classification-System


In [3]:

# ==========================================
# 2. Load Configuration
# ==========================================
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

SPLITS_CSV = Path(cfg["data"]["splits_csv"]) 
NUM_CLASSES = cfg["data"]["num_classes"]
BATCH_SIZE = 32 
LR = 1e-4



In [4]:
# ==========================================
# 3. Custom Dataset
# ==========================================
class WasteDataset(Dataset):
    def __init__(self, csv_file, split='train', transform=None):
        df = pd.read_csv(csv_file)
        # Filter for the specific split (train, val, or test)
        self.data = df[df['split'] == split].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.loc[idx, 'filepath']
        label = self.data.loc[idx, 'class_idx']
        
        # Convert to RGB to ensure all images have 3 channels
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label


In [5]:

# ==========================================
# 4. Augmentations & Transforms
# ==========================================
# EfficientNet-B3 expects 300x300 input resolution
train_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [6]:

# ==========================================
# 5. Data Loaders
# ==========================================
train_ds = WasteDataset(SPLITS_CSV, split='train', transform=train_transform)
val_ds   = WasteDataset(SPLITS_CSV, split='val', transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


In [7]:

# ==========================================
# 6. Model Definition (EfficientNet-B3)
# ==========================================
def get_model(num_classes):
    # Load pre-trained weights for EfficientNet-B3
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
    
    # Replace the classifier head (dropout + linear)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_model(NUM_CLASSES).to(device)


In [8]:

# ==========================================
# 7. Loss and Optimizer
# ==========================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


In [9]:

# ==========================================
# 8. Training Loop
# ==========================================
def train_one_epoch():
    model.train()
    running_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Optional: Print progress every 50 batches
        if (batch_idx + 1) % 50 == 0:
            print(f"  Batch {batch_idx + 1}/{len(train_loader)} - Loss: {loss.item():.4f}")
            
    return running_loss / len(train_loader)

if __name__ == "__main__":
    print(f"🚀 Starting training on {device}...")
    print(f"Training images: {len(train_ds)} | Validation images: {len(val_ds)}")
    
    epochs = 3
    for epoch in range(1, epochs + 1):
        print(f"\n--- Epoch {epoch}/{epochs} ---")
        epoch_loss = train_one_epoch()
        print(f"Epoch {epoch} Average Loss: {epoch_loss:.4f}")

🚀 Starting training on cpu...
Training images: 10500 | Validation images: 2250

--- Epoch 1/3 ---
  Batch 50/329 - Loss: 1.1107
  Batch 100/329 - Loss: 0.6511


KeyboardInterrupt: 